# Imperial Dade — Category Substitution Pipeline (parameterized)

Single notebook for **every category**. Driven by Papermill: the first code cell is tagged `parameters` and is overridden at execution time (e.g. `papermill ... -p category cutlery`).

All category-specific values come from `src/imperial_dade/categories/<category>.yaml`.

In [1]:
# Papermill injects overrides for these in a cell below at runtime.
category = "cups"
stage = "all"  # one of: taxonomy, matching, feedback, optimization, report, all

In [2]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv

# Load .env from repo root (two levels up from this notebook)
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
if (repo_root / '.env').exists():
    load_dotenv(repo_root / '.env')

logging.basicConfig(
    level=os.getenv('IMPERIAL_DADE_LOG_LEVEL', 'INFO'),
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True,
)
log = logging.getLogger('pipeline')
log.info('Pipeline notebook starting — category=%s stage=%s', category, stage)

2026-05-13 18:45:56 [INFO] pipeline: Pipeline notebook starting — category=cups stage=all


In [ ]:
from imperial_dade.categories import load_category
from imperial_dade.config import get_settings, PipelineConfig

cfg = load_category(category)
settings = get_settings()

# Where every stage writes its outputs (mirrors legacy: Data/<Category>/Output/)
output_dir = (settings.data_dir / cfg.data_dir / 'Output').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

log.info('Loaded %s: %d critical attrs, top_n=%d, max_suppliers=%d',
         cfg.name, len(cfg.matching.critical_attributes),
         cfg.matching.top_n, cfg.optimization.max_suppliers)
log.info('Output dir: %s', output_dir)

## Stage 1 — Taxonomy (load + classify)

In [ ]:
if stage in ('taxonomy', 'all'):
    log.info('Stage 1: taxonomy — loading from Fornax + Fabric')
    from imperial_dade.io.fornax import FornaxLoader
    from imperial_dade.io.fabric import FabricLoader
    from imperial_dade.stages import taxonomy_load, taxonomy_classify
    from imperial_dade.llm.client import OpenAIAgent

    # --- Load phase -----------------------------------------------------
    with FabricLoader() as fabric_loader:
        item_segment = fabric_loader.get_item_segment_mapping(branch_company_code='1')
        salsify_bridge = fabric_loader.get_salsify_to_s2k_mapping(entity_id=1)

    with FornaxLoader() as loader:
        sfy             = loader.get_salsify_items(salsify_to_s2k=salsify_bridge)
        item_master_raw = loader.get_consolidated_items(entity_id=1)

        # Find the (division-class) segment keys for THIS category
        segment_keys = item_segment.loc[
            item_segment['Item Segment'] == cfg.name,
            'Item Segment Key'
        ].dropna().unique().tolist()
        log.info('Category %s maps to %d segment keys', cfg.name, len(segment_keys))

        # Item-master filtered to those segment keys gives us the category items
        item_master = item_master_raw[
            item_master_raw['Item Segment Key'].isin(segment_keys)
        ].copy()
        log.info('Item-master filtered to %d items in segment %r', len(item_master), cfg.name)

        # Pull sales for just those item codes
        category_item_codes = item_master['Item Code'].dropna().unique().tolist()
        cat_data = loader.get_sales_data(category_item_codes, cfg, entity_id=1)

    # --- Transform phase ------------------------------------------------
    cat_data_grouped = taxonomy_load.group_data(cat_data)
    _, columns_with_coverage, _ = taxonomy_load.get_columns_with_coverage(
        cat_data_grouped, sfy, cfg.taxonomy.coverage_threshold,
    )
    # Use the YAML-pinned list if provided; else fall back to coverage-discovered
    columns_for_description = cfg.taxonomy.columns_for_description or columns_with_coverage
    log.info('Using %d attribute columns: %s',
             len(columns_for_description), columns_for_description[:5])

    cat_data_final = taxonomy_load.merge_with_salsify(
        cat_data_grouped, sfy, columns_for_description,
    )

    # --- Classify phase -------------------------------------------------
    agent = OpenAIAgent(model='gpt-4.1', chunk_size=32)
    cat_data_final_attributed, taxonomy_df = taxonomy_classify.run(
        cat_data_final,
        agent,
        cfg,
        columns_for_description,
        output_dir=output_dir,
    )
    log.info('Stage 1: taxonomy — complete (%d items)', len(cat_data_final_attributed))
else:
    log.info('skipping taxonomy stage')

## Stage 2 — Matching

In [ ]:
if stage in ('matching', 'all'):
    log.info('Stage 2: matching (top_n=%d)', cfg.matching.top_n)
    from imperial_dade.stages import matching
    from imperial_dade.llm.client import OpenAIAgent

    # If stage='matching' was run alone, the taxonomy CSV is the source of truth.
    if 'cat_data_final_attributed' not in globals():
        import pandas as pd
        attributed_csv = output_dir / f'{cfg.name}_Attributed.csv'
        log.info('Loading taxonomy output from %s', attributed_csv)
        cat_data_final_attributed = pd.read_csv(attributed_csv)

    agent = OpenAIAgent(model='gpt-4o-batch', chunk_size=32)
    im_final = matching.run(
        cat_data_final_attributed,
        agent,
        category=cfg,
        output_dir=output_dir,
        batch_model=False,    # set True for the large async batch endpoint
        pl=True,
        enable_vpn_exclusion=True,
    )
    log.info('Stage 2: matching — complete (%d items)', len(im_final))
else:
    log.info('skipping matching stage')

## Stage 3 — Feedback

In [ ]:
if stage in ('feedback', 'all'):
    log.info('Stage 3: feedback (accept=%r reject=%r)',
             cfg.feedback.accept_label, cfg.feedback.reject_label)
    from imperial_dade.stages import feedback
    from imperial_dade.llm.client import OpenAIAgent

    # If running feedback alone, reload the matching output
    if 'im_final' not in globals():
        import pandas as pd
        matches_csv = output_dir / f'{cfg.name}_matches.csv'
        log.info('Loading matching output from %s', matches_csv)
        im_final = pd.read_csv(matches_csv)

    # Feedback Excel files the analyst has marked up. By convention these
    # land under the same Output/ dir as the Subs_<date>.xlsx the matching
    # stage wrote (renamed with `_with_Feedback`).
    feedback_files = sorted(map(str, output_dir.glob(f'{cfg.name}_Subs*_with_Feedback.xlsx')))
    if not feedback_files:
        raise FileNotFoundError(
            f'No feedback files found under {output_dir}. Expected one or more files matching '
            f'{cfg.name}_Subs*_with_Feedback.xlsx. Mark up the matching-stage output, save with '
            'that name, and re-run this cell.'
        )
    log.info('Found %d feedback file(s): %s', len(feedback_files), feedback_files)

    agent = OpenAIAgent(model='gpt-4o-batch', chunk_size=32)
    hard_rules = (
        "If mentioned, size/volume should be a match if not very close. "
        "Same with color, shape, etc. You should not be trying to swap a 20 oz "
        "product with a 16 oz product."
    )
    im_final2, im_final_with_feedback = feedback.run(
        im_final,
        feedback_files,
        agent,
        category=cfg,
        output_dir=output_dir,
        hard_rules=hard_rules,
        batch_model=False,
    )
    log.info('Stage 3: feedback — complete')
else:
    log.info('skipping feedback stage')

## Stage 4 — Optimization

In [ ]:
if stage in ('optimization', 'all'):
    log.info('Stage: optimization (max_suppliers=%d)', cfg.optimization.max_suppliers)
    from imperial_dade.stages import optimization
    log.warning('TODO: port solve_swap_optimization with max_suppliers=cfg.optimization.max_suppliers and extra_vendor_exclusions=cfg.optimization.extra_vendor_exclusions')
else:
    log.info('skipping optimization stage')

## Stage 5 — Report

In [ ]:
if stage in ('report', 'all'):
    log.info('Stage: report')
    from imperial_dade.stages import report
    log.warning('TODO: port create_enhanced_sku_report; output path = settings.data_dir / cfg.data_dir / Output / <category>_Final_Report.xlsx')
else:
    log.info('skipping report stage')

log.info('Pipeline complete: %s', category)